# Netflix Data Pipeline — Bronze → Silver → Gold


Refatora um notebook de exploração do Kaggle em um pipeline com arquitetura medallion, demonstrando:

- Ingestão fiel da fonte (Bronze) com metadados de rastreabilidade
- Limpeza, tipagem e modelagem dimensional (Silver)
- Agregações de negócio prontas para dashboard (Gold)
- Validações automáticas de qualidade em todas as camadas

## Arquitetura

```
CSV (fonte) → Bronze (cópia + hash) → Silver (limpo + dim) → Gold (métricas) → Dashboard
```

## Por que medallion?

Separar as camadas garante que se algo quebrar no Silver (ex.: nova regra de limpeza), o Bronze ainda preserva a fonte intacta para reprocessar — sem precisar baixar o CSV de novo. Cada camada tem uma responsabilidade única.

## 1. Setup

In [ ]:
!pip install -q pandas pyarrow

In [ ]:
import pandas as pd
import hashlib
import json
from pathlib import Path
from datetime import datetime

BASE = Path('data')
for layer in ['bronze', 'silver', 'gold']:
    (BASE / layer).mkdir(parents=True, exist_ok=True)

CATALOG = BASE / '_catalog.jsonl'

## 2. Upload do CSV

Selecione `netflix_titles.csv` (Kaggle: shivamb/netflix-shows).

In [ ]:
from google.colab import files
uploaded = files.upload()
SOURCE_PATH = next(iter(uploaded.keys()))
print(f'Fonte: {SOURCE_PATH}')

## 3. Catálogo de metadados

Em produção isso seria Glue/Unity/DataHub. Aqui só registra em JSONL local — o importante é demonstrar que **toda execução é rastreada**: quem rodou, quando, quantas linhas, qual hash da fonte.

In [ ]:
def register_metadata(layer, table, row_count, **extras):
    entry = {
        'layer': layer,
        'table': table,
        'row_count': row_count,
        'registered_at': datetime.utcnow().isoformat(),
        **extras,
    }
    with CATALOG.open('a') as f:
        f.write(json.dumps(entry) + '\n')
    print(f'Catálogo: {entry}')

## 4. Camada BRONZE — Ingestão

**Objetivo:** copiar a fonte sem alterar nada + adicionar metadados de rastreabilidade.

Decisões importantes:

- `dtype=str` ao ler — não confiar na inferência do pandas; tipagem é problema do Silver.
- Hash SHA-256 do arquivo — permite detectar se a fonte mudou entre execuções (idempotência).
- Particionar por `_partition_date` — facilita ler só dados recentes em produção.
- Parquet + snappy — ~10x menor que CSV e muito mais rápido de ler.

In [ ]:
def ingest_to_bronze(source_path: str) -> pd.DataFrame:
    df = pd.read_csv(source_path, dtype=str)

    now = datetime.utcnow()
    df['_source_file'] = source_path
    df['_ingested_at'] = now
    df['_partition_date'] = now.date()

    with open(source_path, 'rb') as f:
        file_hash = hashlib.sha256(f.read()).hexdigest()
    df['_source_hash'] = file_hash

    output = BASE / 'bronze' / 'netflix_titles'
    df.to_parquet(
        output,
        partition_cols=['_partition_date'],
        index=False,
        compression='snappy',
    )

    print(f'Bronze: {len(df)} registros em {output}')
    register_metadata('bronze', 'netflix_titles', len(df), source_hash=file_hash)
    return df

df_bronze = ingest_to_bronze(SOURCE_PATH)
df_bronze.head()

## 5. Camada SILVER — Limpeza + Modelo Dimensional

**Objetivo:** corrigir problemas de qualidade do Bronze e quebrar em fato + dimensões.

Problemas resolvidos aqui (todos vieram do notebook original do Kaggle):

1. **Multi-valores tratados como string única** — `country = "United States, India"` contava como 1 país. Solução: `str.split` + `explode` → cada país vira uma linha (`dim_pais`).
2. **`fillna('Unknown')` distorcia rankings** — "Unknown" virava o diretor mais frequente. Solução: filtrar nulos no `groupby`, não preencher.
3. **`duration` mistura unidades** — "90 min" (filme) e "2 Seasons" (série) na mesma coluna. Solução: regex para separar `duration_value` (int) e `duration_unit` (string).
4. **Tipos errados** — `release_year` como string, `date_added` como string. Solução: `to_datetime` / `to_numeric` com `errors='coerce'`.

**Validações pré-transformação** (asserts) garantem contratos da fonte; **validações pós** garantem que a transformação não introduziu inconsistências.

In [ ]:
def transform_to_silver() -> dict:
    bronze_path = BASE / 'bronze' / 'netflix_titles'
    parquet_files = list(bronze_path.rglob('*.parquet'))
    if not parquet_files:
        raise FileNotFoundError(f'Nenhum Parquet em {bronze_path}. Rode o Bronze primeiro.')

    df = pd.read_parquet(parquet_files, engine='pyarrow')
    print(f'Lidos {len(df)} registros do Bronze')

    df = df.drop_duplicates(subset=['show_id'])
    print(f'Após dedup: {len(df)} registros')

    assert df['show_id'].is_unique
    assert df['type'].isin(['Movie', 'TV Show']).all()

    df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
    df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce')

    parsed = df['duration'].str.extract(r'(?P<v>\d+)\s*(?P<u>min|Season|Seasons)')
    df['duration_value'] = pd.to_numeric(parsed['v'])
    df['duration_unit'] = parsed['u'].replace({'Seasons': 'Season'})

    fato_titulos = df[['show_id','type','title','director','release_year','rating',
                       'date_added','duration_value','duration_unit','description']].copy()

    def explodir(col_origem: str, col_destino: str) -> pd.DataFrame:
        """Quebra string multi-valor em linhas. Tolera ',', ', ', ' ,', ' , '."""
        out = (
            df[['show_id', col_origem]]
            .dropna(subset=[col_origem])
            .assign(**{col_destino: lambda d: d[col_origem].str.split(r'\s*,\s*', regex=True)})
            .explode(col_destino)[['show_id', col_destino]]
        )
        out[col_destino] = out[col_destino].astype(str).str.strip()
        out = out[out[col_destino].ne('') & out[col_destino].ne('nan')]
        return out.drop_duplicates()

    dim_pais = explodir('country', 'country')
    dim_genero = explodir('listed_in', 'genre')
    dim_elenco = explodir('cast', 'actor')

    bad = dim_pais[dim_pais['country'].str.contains(',', na=False)]
    if not bad.empty:
        print('!!! Valores com vírgula em dim_pais (debug):')
        print(bad.head(10))
        raise AssertionError('explode falhou — ver valores acima')

    validate_silver(fato_titulos, dim_pais, dim_genero)

    tables = {
        'fato_titulos': fato_titulos,
        'dim_pais': dim_pais,
        'dim_genero': dim_genero,
        'dim_elenco': dim_elenco,
    }
    for nome, tabela in tables.items():
        path = BASE / 'silver' / f'{nome}.parquet'
        tabela.to_parquet(path, index=False)
        print(f'Silver: {nome} -> {len(tabela)} registros')
        register_metadata('silver', nome, len(tabela))
    return tables


def validate_silver(fato, dim_pais, dim_genero):
    assert fato['show_id'].is_unique, 'fato_titulos.show_id não é único'
    assert fato['release_year'].dropna().between(1900, 2030).all()
    assert fato['type'].isin(['Movie', 'TV Show']).all()
    assert dim_pais['show_id'].isin(fato['show_id']).all()
    assert not dim_pais['country'].str.contains(',', na=False).any(), 'vírgula em dim_pais'
    assert dim_genero['show_id'].isin(fato['show_id']).all()
    print('Validações Silver: OK')


# Limpa Silver/Gold antigos para garantir reescrita.
import shutil
for layer in ['silver', 'gold']:
    p = BASE / layer
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

silver = transform_to_silver()


## 6. Camada GOLD — Métricas de Negócio

**Objetivo:** agregações prontas para alimentar dashboards. Cada tabela Gold corresponde a um gráfico.

Métricas que respondem ao pedido típico de negócio: *"quais países produzem mais conteúdo, como o catálogo evoluiu, quais gêneros dominam, e qual a proporção filme/série?"*

In [ ]:
def create_gold_metrics() -> dict:
    fato = pd.read_parquet(BASE / 'silver' / 'fato_titulos.parquet')
    dim_pais = pd.read_parquet(BASE / 'silver' / 'dim_pais.parquet')
    dim_genero = pd.read_parquet(BASE / 'silver' / 'dim_genero.parquet')

    total_titulos = fato['show_id'].nunique()

    gold_por_pais = (
        dim_pais
        .groupby('country', as_index=False)
        .agg(qtd_titulos=('show_id', 'nunique'))
        .assign(pct_total=lambda d: (100 * d['qtd_titulos'] / total_titulos).round(2))
        .sort_values('qtd_titulos', ascending=False)
        .head(20)
    )

    gold_adicoes_tempo = (
        fato
        .dropna(subset=['date_added'])
        .assign(
            ano=lambda d: d['date_added'].dt.year,
            mes=lambda d: d['date_added'].dt.month,
        )
        .groupby(['ano', 'mes', 'type'], as_index=False)
        .agg(qtd_titulos=('show_id', 'nunique'))
    )

    gold_generos = (
        dim_genero
        .groupby('genre', as_index=False)
        .agg(qtd_titulos=('show_id', 'nunique'))
        .sort_values('qtd_titulos', ascending=False)
        .head(15)
    )

    gold_tipo = (
        fato
        .groupby('type', as_index=False)
        .agg(
            qtd_titulos=('show_id', 'nunique'),
            duracao_media=('duration_value', 'mean'),
        )
    )

    metrics = {
        'gold_por_pais': gold_por_pais,
        'gold_adicoes_tempo': gold_adicoes_tempo,
        'gold_generos': gold_generos,
        'gold_tipo': gold_tipo,
    }
    for nome, tabela in metrics.items():
        path = BASE / 'gold' / f'{nome}.parquet'
        tabela.to_parquet(path, index=False)
        print(f'Gold: {nome} -> {len(tabela)} registros')
        register_metadata('gold', nome, len(tabela))
    return metrics

gold = create_gold_metrics()

## 7. Inspeção das tabelas Gold

In [ ]:
print('Top 10 países')
display(gold['gold_por_pais'].head(10))

print('Filmes vs Séries')
display(gold['gold_tipo'])

print('Top 10 gêneros')
display(gold['gold_generos'].head(10))

## 8. Relatório de qualidade

Função típica do dia a dia de Analista de Engenharia de Dados: gerar relatório semanal de saúde dos dados e alertar quando algum threshold é violado.

In [ ]:
def relatorio_qualidade():
    df = pd.read_parquet(BASE / 'silver' / 'fato_titulos.parquet')
    total = len(df)
    rel = {
        'total_registros': total,
        'duplicatas_show_id': int(df.duplicated(subset='show_id').sum()),
        'nulos_title': int(df['title'].isna().sum()),
        'nulos_release_year': int(df['release_year'].isna().sum()),
        'nulos_date_added': int(df['date_added'].isna().sum()),
        'nulos_director': int(df['director'].isna().sum()),
        'pct_filmes': round(100 * (df['type'] == 'Movie').sum() / total, 2),
        'pct_series': round(100 * (df['type'] == 'TV Show').sum() / total, 2),
    }
    return rel

relatorio_qualidade()

## 9. Linhagem & dicionário (resumo)

**Linhagem:** `netflix_titles.csv` → `bronze/netflix_titles` → `silver/{fato_titulos, dim_pais, dim_genero, dim_elenco}` → `gold/{por_pais, adicoes_tempo, generos, tipo}`

**fato_titulos** (1 linha por título):

| Campo | Tipo | Regra |
|-------|------|-------|
| show_id | string | PK, único, não nulo |
| type | string | domínio: Movie ou TV Show |
| title | string | não nulo |
| release_year | int | entre 1900 e 2030 |
| date_added | datetime | pode ser nulo (catálogo histórico) |
| duration_value | int | minutos (filme) ou temporadas (série) |
| duration_unit | string | depende de `type` |

**dim_pais / dim_genero / dim_elenco** (FK = show_id, granularidade = 1 valor por linha após explode).

